# Module 06: AI in Retail - GenAI-Powered Product Description Generator
**Author:** Sergio Sanchez  
**Course:** 6271-ITAI-2372-Artificial Intel Applications  
**Topic:** Instruction-Tuned Text Generation, Prompt Engineering, & Multi-Tone Personalization  

---

## 1. Executive Summary & Retail Context
In modern retail and e-commerce, static catalog copy struggles to resonate across diverse demographic cohorts. E-commerce platforms leverage generative AI to personalize product marketing, tailor tone, and adapt messaging for specific customer segments without manually rewriting thousands of SKUs.

### Core Objectives
1. **Input Conditioning:** Ingest product metadata, verified features, intended audience, and writing tone.
2. **Instruction-Tuned Local LLM:** Deploy a local open-source transformer (`google/flan-t5-base`) utilizing zero external API dependencies or fees.
3. **Multi-Tone Personalization (Stretch Goal):** Implement stylistic control allowing dynamic generation across Professional, Casual, Humorous, and Youth-oriented personas.
4. **Factuality & Anti-Hallucination Guardrails:** Design system-level prompt constraints to prevent the model from inventing unverified specifications, warranties, or material claims.

## 2. Environment Setup & Model Loading
We load Hugging Face's `AutoTokenizer` and `AutoModelForSeq2SeqLM` to initialize `google/flan-t5-base`. Flan-T5 is explicitly instruction-tuned, making it well-suited for following strict copy generation constraints on local hardware without requiring paid API keys.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Select device (GPU if available, otherwise CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing pipeline on device: {device}")

# Load instruction-tuned model and tokenizer
model_id = "google/flan-t5-base"
print(f"Loading {model_id} from Hugging Face...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

print("Instruction-tuned generator initialized successfully.")

C:\Users\solut\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Executing pipeline on device: cpu
Loading google/flan-t5-base from Hugging Face...


C:\Users\solut\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\solut\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\solut\AppData\Local

Instruction-tuned generator initialized successfully.


## 3. Structured Prompt Engineering & Tone Steering
To ensure marketing copy remains compelling while adhering strictly to documented product facts, the prompt incorporates:
- **Task Identity:** Directing the model to act as an e-commerce copywriter.
- **Stylistic Constraints:** Mapping selected tone and target audience into concrete writing guidelines.
- **Negative Constraints:** Prohibiting invented claims, unverified materials, or fabricated certifications.

In [2]:
TONE_DIRECTIVES = {
    "formal": "Professional, precise, polished, and structured for enterprise or discerning buyers.",
    "casual": "Friendly, warm, conversational, and approachable for everyday shoppers.",
    "humorous": "Lighthearted, playful, and engaging with witty phrasing.",
    "young_adults": "Energetic, dynamic, modern, and relatable for active youth and young adults.",
    "professionals": "Concise, data-oriented, credible, and focused on technical reliability."
}

def generate_product_description(
    product_name: str,
    keywords: list,
    audience: str = "General Shoppers",
    tone: str = "casual",
    max_length: int = 150
) -> str:
    """
    Generates tailored e-commerce copy using local instruction-tuned Flan-T5.
    Enforces strict factuality guardrails to prevent ungrounded hallucinations.
    """
    tone_key = tone.lower().strip()
    tone_guide = TONE_DIRECTIVES.get(tone_key, TONE_DIRECTIVES["casual"])
    features_str = ", ".join(keywords)
    
    # Construct structured prompt with factuality constraints
    prompt = (
        f"Instruction: You are an e-commerce copywriter. Write a compelling product description.\n"
        f"Product Name: {product_name}\n"
        f"Verified Features: {features_str}\n"
        f"Target Audience: {audience}\n"
        f"Desired Tone: {tone_key} ({tone_guide})\n"
        f"Constraint: Use only the verified features listed above. Do not invent unverified claims, "
        f"certifications, or specifications. Write directly to the customer in complete sentences."
    )
    
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(device)
    
    output_tokens = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        min_length=25,
        num_beams=4,
        no_repeat_ngram_size=2,
        early_stopping=True
    )
    
    description = tokenizer.decode(output_tokens[0], skip_special_tokens=True).strip()
    return description

print("Generator function registered with multi-tone support.")

Generator function registered with multi-tone support.


## 4. Multi-Segment Testing Suite & Experimental Evaluation
We run controlled experiments against identical product features across different target audiences and tones (Professional, Casual, Young Adults, and Humorous) to evaluate personalization fidelity and ground-truth constraint adherence.

In [3]:
test_catalog = [
    {
        "product_name": "Apex Pro Wireless Mechanical Keyboard",
        "keywords": ["hot-swappable tactile switches", "aircraft-grade aluminum frame", "RGB per-key backlighting", "40-hour rechargeable battery", "Bluetooth 5.2 connectivity"],
        "audience": "Software Engineers and Remote Professionals",
        "tone": "professionals"
    },
    {
        "product_name": "Apex Pro Wireless Mechanical Keyboard",
        "keywords": ["hot-swappable tactile switches", "aircraft-grade aluminum frame", "RGB per-key backlighting", "40-hour rechargeable battery", "Bluetooth 5.2 connectivity"],
        "audience": "College Gamers & Streamers",
        "tone": "young_adults"
    },
    {
        "product_name": "EcoTrail Insulated Hydration Flask",
        "keywords": ["double-wall vacuum insulation", "recycled 18/8 stainless steel", "leak-proof spout lid", "24-hour cold retention", "sweat-proof powder coat"],
        "audience": "Casual Day Hikers and Fitness Walkers",
        "tone": "casual"
    },
    {
        "product_name": "EcoTrail Insulated Hydration Flask",
        "keywords": ["double-wall vacuum insulation", "recycled 18/8 stainless steel", "leak-proof spout lid", "24-hour cold retention", "sweat-proof powder coat"],
        "audience": "Exhausted Desk Workers Needing Hydration",
        "tone": "humorous"
    }
]

print("=" * 85)
print("RETAIL GENAI PRODUCT DESCRIPTION GENERATOR - INFERENCE RESULTS")
print("=" * 85)

for idx, item in enumerate(test_catalog, start=1):
    generated_copy = generate_product_description(
        product_name=item["product_name"],
        keywords=item["keywords"],
        audience=item["audience"],
        tone=item["tone"]
    )
    
    print(f"\n[EXPERIMENT {idx}] {item['product_name'].upper()}")
    print("-" * 85)
    print(f"• Target Audience : {item['audience']}")
    print(f"• Writing Tone    : {item['tone'].upper()}")
    print(f"• Features Input  : {', '.join(item['keywords'])}")
    print(f"• Generated Copy  : {generated_copy}")

print("\n" + "=" * 85)

RETAIL GENAI PRODUCT DESCRIPTION GENERATOR - INFERENCE RESULTS

[EXPERIMENT 1] APEX PRO WIRELESS MECHANICAL KEYBOARD
-------------------------------------------------------------------------------------
• Target Audience : Software Engineers and Remote Professionals
• Writing Tone    : PROFESSIONALS
• Features Input  : hot-swappable tactile switches, aircraft-grade aluminum frame, RGB per-key backlighting, 40-hour rechargeable battery, Bluetooth 5.2 connectivity
• Generated Copy  : Apex Pro Wireless Mechanical Keyboard Verified Features: hot-swappable tactile switches, aircraft-grade aluminum frame, RGB per-key backlighting, 40-hour rechargeable battery, Bluetooth 5.2 connectivity

[EXPERIMENT 2] APEX PRO WIRELESS MECHANICAL KEYBOARD
-------------------------------------------------------------------------------------
• Target Audience : College Gamers & Streamers
• Writing Tone    : YOUNG_ADULTS
• Features Input  : hot-swappable tactile switches, aircraft-grade aluminum frame, RGB per

## 5. Architectural Evaluation, Retail Strategy, & Ethical Governance

### Architecture Selection & Trade-Off Analysis
For this implementation, an open-weights, instruction-tuned Seq2Seq model (`google/flan-t5-base`) running locally via Hugging Face was selected over hosted proprietary APIs (such as OpenAI Davinci/GPT-4). This decision prioritizes:
1. **Operational Privacy & Sovereignty:** Unreleased product specifications, internal supplier codes, and trade secrets remain strictly on local infrastructure, eliminating exposure through third-party cloud data-retention pipelines.
2. **Cost Predictability & Zero Token Latency:** Hosted APIs introduce per-token billing overhead and rate limits that become economically prohibitive when scaling across an enterprise catalog of 50,000+ SKUs.
3. **Instruction Compliance:** Unlike raw autoregressive models (e.g., standard GPT-2), Flan-T5 is explicitly fine-tuned on instruction-following datasets, enabling predictable adherence to negative constraints and tone steering without extensive prompt bloating.

### Challenges Encountered
- **Vocabulary Hallucination Control:** Standard generative models frequently inject unsupported superlatives (e.g., "certified waterproof" or "military grade") when attempting to make copy sound appealing. Enforcing explicit prompt boundaries (`"Use only the verified features listed above"`) alongside beam search decoding (`num_beams=4`) was essential to keep outputs grounded.
- **Balancing Persona Stylization with Factuality:** When pushed into stylized personas (e.g., "humorous"), smaller models risk drifting from core product specifications to satisfy stylistic constraints. Strict task framing resolved this tension.

### Connection to Retail Systems & Ethical Considerations
- **The "Phantom Stockout" Connection:** As demonstrated in the Larkfield & Co. case study, automated systems fail when decoupled from physical reality. In retail content generation, deploying autonomous text models without human oversight risks false-advertising liabilities, deceptive claims, and customer trust erosion.
- **Governance Safeguard:** Generative AI in retail must operate on **Rung 4 of the Autonomy Ladder (Proposes, You Approve)**. Marketing specialists must retain final editorial sign-off to ensure generated promotional claims strictly reflect verified supplier specifications.